# ECE 3715 — Mini-Project 2
## Continuous Random Variables

### Team Members
- Christian Ruelas
- Partner 1
- Partner 2

### Contributions
- Christian: Steps 1, 2, and 4
- Partner 1: Steps 3 and 5
- Partner 2: Steps 6, 7, and 8


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm

from mp2lib import (
    inv_cdf_exponential,
    inv_cdf_rayleigh,
    sample_by_inversion,
    ks_statistic,
    q_function,
)


In [ ]:
# Fixed team seed -- DO NOT CHANGE
SEED = 611441029

rng = np.random.default_rng(SEED)

# Independent generators for each step
rng1, rng2, rng3, rng4, rng5, rng6, rng7, rng8 = rng.spawn(8)

print(f"Team seed: {SEED}")
print(f"NumPy version: {np.__version__}")


# Step 1 — From CDF to Density

We consider four continuous random variables: Uniform, Exponential, Gaussian, and Rayleigh.

## Uniform Distribution

For \(X \sim U(0,1)\),

\[
F_X(x)=
\begin{cases}
0, & x < 0\\
x, & 0 \le x \le 1\\
1, & x > 1
\end{cases}
\]

and

\[
f_X(x)=
\begin{cases}
1, & 0 \le x \le 1\\
0, & \text{otherwise}.
\end{cases}
\]

The support is \(0 \le x \le 1\).

## Exponential Distribution

For \(X \sim \mathrm{Exp}(\lambda)\), with \(\lambda=1\),

\[
F_X(x)=
\begin{cases}
0, & x < 0\\
1-e^{-\lambda x}, & x \ge 0
\end{cases}
\]

and

\[
f_X(x)=
\begin{cases}
\lambda e^{-\lambda x}, & x \ge 0\\
0, & x < 0.
\end{cases}
\]

The support is \(x \ge 0\).

## Standard Gaussian Distribution

For \(X \sim \mathcal{N}(0,1)\),

\[
F_X(x)=\Phi(x)
\]

and

\[
f_X(x)=\frac{1}{\sqrt{2\pi}}e^{-x^2/2}.
\]

The support is \(-\infty < x < \infty\).

## Rayleigh Distribution

For \(X \sim \mathrm{Rayleigh}(\sigma)\), with \(\sigma=1\),

\[
F_X(x)=
\begin{cases}
0, & x < 0\\
1-e^{-x^2/(2\sigma^2)}, & x \ge 0
\end{cases}
\]

and

\[
f_X(x)=
\begin{cases}
\frac{x}{\sigma^2}e^{-x^2/(2\sigma^2)}, & x \ge 0\\
0, & x < 0.
\end{cases}
\]

The support is \(x \ge 0\).


In [ ]:
# Step 1 parameters
N1 = 100_000
lam = 1.0
sigma = 1.0

# Library generators are allowed in Step 1.
uniform_samples = rng1.uniform(0.0, 1.0, N1)
exponential_samples = rng1.exponential(scale=1.0 / lam, size=N1)
gaussian_samples = rng1.normal(loc=0.0, scale=1.0, size=N1)
rayleigh_samples = rng1.rayleigh(scale=sigma, size=N1)

print(f"Generated {N1:,} samples for each Step 1 distribution.")


In [ ]:
def uniform_cdf(x):
    x = np.asarray(x)
    return np.where(x < 0.0, 0.0, np.where(x <= 1.0, x, 1.0))


def uniform_pdf(x):
    x = np.asarray(x)
    return np.where((x >= 0.0) & (x <= 1.0), 1.0, 0.0)


def exponential_cdf(x, lam=1.0):
    x = np.asarray(x)
    return np.where(x < 0.0, 0.0, 1.0 - np.exp(-lam * x))


def exponential_pdf(x, lam=1.0):
    x = np.asarray(x)
    return np.where(x < 0.0, 0.0, lam * np.exp(-lam * x))


def gaussian_cdf(x):
    return norm.cdf(x)


def gaussian_pdf(x):
    return norm.pdf(x)


def rayleigh_cdf(x, sigma=1.0):
    x = np.asarray(x)
    return np.where(
        x < 0.0,
        0.0,
        1.0 - np.exp(-(x**2) / (2.0 * sigma**2)),
    )


def rayleigh_pdf(x, sigma=1.0):
    x = np.asarray(x)
    return np.where(
        x < 0.0,
        0.0,
        (x / sigma**2) * np.exp(-(x**2) / (2.0 * sigma**2)),
    )


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

distributions = [
    ("Uniform", uniform_samples, uniform_cdf, uniform_pdf, np.linspace(0, 1, 500)),
    ("Exponential", exponential_samples, exponential_cdf, exponential_pdf, np.linspace(0, 8, 500)),
    ("Gaussian", gaussian_samples, gaussian_cdf, gaussian_pdf, np.linspace(-4, 4, 500)),
    ("Rayleigh", rayleigh_samples, rayleigh_cdf, rayleigh_pdf, np.linspace(0, 5, 500)),
]

for col, (name, samples, cdf, pdf, x_grid) in enumerate(distributions):
    sorted_samples = np.sort(samples)
    empirical_cdf = np.arange(1, len(samples) + 1) / len(samples)

    axes[0, col].plot(sorted_samples, empirical_cdf, label="Empirical CDF")
    axes[0, col].plot(x_grid, cdf(x_grid), "--", label="Analytic CDF")
    axes[0, col].set_title(f"{name} CDF")
    axes[0, col].set_xlabel("x")
    axes[0, col].set_ylabel("Probability")
    axes[0, col].grid(True)
    axes[0, col].legend()

    axes[1, col].hist(samples, bins=80, density=True, alpha=0.6, label="Sample Histogram")
    axes[1, col].plot(x_grid, pdf(x_grid), linewidth=2, label="Analytic PDF")
    axes[1, col].set_title(f"{name} Density")
    axes[1, col].set_xlabel("x")
    axes[1, col].set_ylabel("Density")
    axes[1, col].grid(True)
    axes[1, col].legend()

plt.suptitle("Figure 1: Empirical and Analytic Continuous Distributions")
plt.tight_layout()
plt.show()


In [ ]:
ks_uniform = ks_statistic(uniform_samples, uniform_cdf)
ks_exponential = ks_statistic(exponential_samples, exponential_cdf)
ks_gaussian = ks_statistic(gaussian_samples, gaussian_cdf)
ks_rayleigh = ks_statistic(rayleigh_samples, rayleigh_cdf)

ks_critical = 1.36 / np.sqrt(N1)

print("Step 1 KS Statistics")
print("-" * 55)
print(f"{'Distribution':<15} {'KS Statistic':<15} {'Critical Value':<15}")
print("-" * 55)
print(f"{'Uniform':<15} {ks_uniform:<15.6f} {ks_critical:<15.6f}")
print(f"{'Exponential':<15} {ks_exponential:<15.6f} {ks_critical:<15.6f}")
print(f"{'Gaussian':<15} {ks_gaussian:<15.6f} {ks_critical:<15.6f}")
print(f"{'Rayleigh':<15} {ks_rayleigh:<15.6f} {ks_critical:<15.6f}")


## Exponential and Rayleigh Moments

For an exponential random variable with rate \(\lambda=1\),

\[
E[X]=\frac{1}{\lambda}=1,
\]

\[
\mathrm{median}(X)=\frac{\ln 2}{\lambda}
=\ln 2\approx0.6931,
\]

and

\[
\mathrm{Var}(X)=\frac{1}{\lambda^2}=1.
\]

For a Rayleigh random variable with scale \(\sigma=1\),

\[
E[X]=\sigma\sqrt{\frac{\pi}{2}}
=\sqrt{\frac{\pi}{2}}
\approx1.2533,
\]

\[
\mathrm{median}(X)
=\sigma\sqrt{2\ln2}
=\sqrt{2\ln2}
\approx1.1774,
\]

and

\[
\mathrm{Var}(X)
=
\frac{4-\pi}{2}\sigma^2
=
\frac{4-\pi}{2}
\approx0.4292.
\]


In [ ]:
exp_mean_theory = 1.0 / lam
exp_median_theory = np.log(2.0) / lam
exp_variance_theory = 1.0 / lam**2

ray_mean_theory = sigma * np.sqrt(np.pi / 2.0)
ray_median_theory = sigma * np.sqrt(2.0 * np.log(2.0))
ray_variance_theory = ((4.0 - np.pi) / 2.0) * sigma**2

print("Exponential Distribution")
print("-" * 50)
print(f"{'Statistic':<12} {'Sample':<15} {'Theory':<15}")
print(f"{'Mean':<12} {np.mean(exponential_samples):<15.6f} {exp_mean_theory:<15.6f}")
print(f"{'Median':<12} {np.median(exponential_samples):<15.6f} {exp_median_theory:<15.6f}")
print(f"{'Variance':<12} {np.var(exponential_samples):<15.6f} {exp_variance_theory:<15.6f}")

print()

print("Rayleigh Distribution")
print("-" * 50)
print(f"{'Statistic':<12} {'Sample':<15} {'Theory':<15}")
print(f"{'Mean':<12} {np.mean(rayleigh_samples):<15.6f} {ray_mean_theory:<15.6f}")
print(f"{'Median':<12} {np.median(rayleigh_samples):<15.6f} {ray_median_theory:<15.6f}")
print(f"{'Variance':<12} {np.var(rayleigh_samples):<15.6f} {ray_variance_theory:<15.6f}")


### Step 1 Interpretation

The empirical CDFs closely follow their corresponding analytic CDFs, and the normalized sample histograms follow the analytic probability densities. This agreement is quantified using the KS statistic rather than relying only on visual inspection.

The exponential distribution has a mean larger than its median because its density is positively skewed and has a long right tail. In contrast, the Gaussian density is symmetric about its mean, so its mean and median coincide.


# Step 2 — Build Your Own Generator by Inversion

## Inverse Transform Sampling

Let

\[
U\sim U(0,1)
\]

and let \(F\) be a continuous, strictly increasing CDF. Define

\[
X=F^{-1}(U).
\]

Then

\[
P(X\le x)=P(F^{-1}(U)\le x)
\]

and because \(F\) is strictly increasing,

\[
P(F^{-1}(U)\le x)=P(U\le F(x))=F(x).
\]

Therefore,

\[
P(X\le x)=F(x),
\]

so \(F^{-1}(U)\) has CDF \(F\).

## Exponential Inverse CDF

The exponential CDF is

\[
u=1-e^{-\lambda x}.
\]

Solving for \(x\),

\[
1-u=e^{-\lambda x},
\]

\[
\ln(1-u)=-\lambda x,
\]

so

\[
\boxed{
F_X^{-1}(u)=-\frac{\ln(1-u)}{\lambda}
}
\]

for \(0\le u<1\).

## Rayleigh Inverse CDF

The Rayleigh CDF is

\[
u=1-e^{-x^2/(2\sigma^2)}.
\]

Thus,

\[
1-u=e^{-x^2/(2\sigma^2)},
\]

\[
\ln(1-u)=-\frac{x^2}{2\sigma^2},
\]

\[
x^2=-2\sigma^2\ln(1-u).
\]

Because a Rayleigh random variable is nonnegative,

\[
\boxed{
F_X^{-1}(u)=\sigma\sqrt{-2\ln(1-u)}
}
\]

for \(0\le u<1\).


In [ ]:
# Simple hand-checkable tests
u_test = np.array([0.0, 0.5])

exp_test = inv_cdf_exponential(u_test, lam=1.0)
ray_test = inv_cdf_rayleigh(u_test, sigma=1.0)

print("Inverse CDF Tests")
print("-----------------")
print("u =", u_test)
print("Exponential =", exp_test)
print("Rayleigh =", ray_test)


In [ ]:
N2 = 100_000

exp_inversion_samples = sample_by_inversion(
    inv_cdf_exponential,
    N2,
    rng2,
    lam=1.0,
)

ray_inversion_samples = sample_by_inversion(
    inv_cdf_rayleigh,
    N2,
    rng2,
    sigma=1.0,
)

print("Inverse-transform samples generated.")
print(f"N = {N2:,}")


In [ ]:
ks_exp_inversion = ks_statistic(
    exp_inversion_samples,
    exponential_cdf,
)

ks_ray_inversion = ks_statistic(
    ray_inversion_samples,
    rayleigh_cdf,
)

ks_critical_step2 = 1.36 / np.sqrt(N2)

print("Step 2 KS Statistics")
print("-" * 55)
print(f"Exponential: KS = {ks_exp_inversion:.6f}, critical = {ks_critical_step2:.6f}")
print(f"Rayleigh:   KS = {ks_ray_inversion:.6f}, critical = {ks_critical_step2:.6f}")


In [ ]:
# Compare against NumPy's built-in generators without disturbing rng2.
# This comparison generator is deterministic and separate from the Step 2 stream.
rng2_compare = np.random.default_rng(SEED + 2)

exp_library_samples = rng2_compare.exponential(
    scale=1.0,
    size=N2,
)

ray_library_samples = rng2_compare.rayleigh(
    scale=1.0,
    size=N2,
)

print("Sample Means")
print("-" * 45)
print(f"Exponential inversion: {np.mean(exp_inversion_samples):.6f}")
print(f"Exponential library:   {np.mean(exp_library_samples):.6f}")
print()
print(f"Rayleigh inversion:    {np.mean(ray_inversion_samples):.6f}")
print(f"Rayleigh library:      {np.mean(ray_library_samples):.6f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x_exp = np.linspace(
    0,
    np.percentile(exp_inversion_samples, 99.9),
    500,
)
sorted_exp = np.sort(exp_inversion_samples)
empirical_exp = np.arange(1, N2 + 1) / N2

axes[0].plot(
    sorted_exp,
    empirical_exp,
    label=f"Inversion samples (KS={ks_exp_inversion:.4f})",
)
axes[0].plot(
    x_exp,
    exponential_cdf(x_exp),
    "--",
    label="Analytic CDF",
)
axes[0].set_title("Exponential Distribution")
axes[0].set_xlabel("x")
axes[0].set_ylabel("CDF")
axes[0].grid(True)
axes[0].legend()

x_ray = np.linspace(
    0,
    np.percentile(ray_inversion_samples, 99.9),
    500,
)
sorted_ray = np.sort(ray_inversion_samples)
empirical_ray = np.arange(1, N2 + 1) / N2

axes[1].plot(
    sorted_ray,
    empirical_ray,
    label=f"Inversion samples (KS={ks_ray_inversion:.4f})",
)
axes[1].plot(
    x_ray,
    rayleigh_cdf(x_ray),
    "--",
    label="Analytic CDF",
)
axes[1].set_title("Rayleigh Distribution")
axes[1].set_xlabel("x")
axes[1].set_ylabel("CDF")
axes[1].grid(True)
axes[1].legend()

plt.suptitle("Figure 2: Inverse Transform Sampling")
plt.tight_layout()
plt.show()


### Edge Case at \(U=0\)

NumPy's `rng.random()` generates values in the interval

\[
0\le U<1.
\]

Therefore, \(U=0\) is possible while \(U=1\) is excluded. For the inverse-CDF formulas used here,

\[
F_{\mathrm{Exp}}^{-1}(0)=0
\]

and

\[
F_{\mathrm{Rayleigh}}^{-1}(0)=0.
\]

Thus, the possible value \(U=0\) does not cause an infinity or numerical failure in this implementation. The code uses `np.log1p(-u)` to evaluate \(\ln(1-u)\) accurately for values of \(u\) near zero.

### Equivalent Exponential Inverse Forms

The exponential sample may be generated using either

\[
-\frac{\ln(1-U)}{\lambda}
\]

or

\[
-\frac{\ln U}{\lambda}.
\]

Both are valid because if \(U\sim U(0,1)\), then \(1-U\) is also uniformly distributed on \((0,1)\). This project uses

\[
-\frac{\ln(1-U)}{\lambda}
\]

because it follows directly from solving the exponential CDF for \(x\).

### Step 2 Interpretation

The empirical CDFs produced by inverse-transform sampling closely follow the analytic exponential and Rayleigh CDFs. The KS statistics quantify that agreement and should fall below the 95% critical value for this sample size.

The sample statistics also agree with NumPy's built-in generators, demonstrating that the distributions can be constructed directly from uniform random numbers and the appropriate inverse CDF.


# Step 3 — Partner 1

Continue the project here.


# Step 4 — Christian

Continue the project here after the Step 3 handoff.


# Step 5 — Partner 1

Continue the project here.


# Steps 6–8 — Partner 2

Continue the detector and operating-point sections here.
